**Import libraries**

In [3]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

**User inputs**

In [ ]:
from pathlib import Path
import sys

# ── User configuration ────────────────────────────────────────────────────────
# Update name_path to your local path before running
# Example (Mac):     "/Users/yourname/Google Drive/My Drive/gogolou_2026_notch-main"
# Example (Windows): "C:/Users/yourname/Google Drive/My Drive/gogolou_2026_notch-main"
# Example (Colab):   "/content/drive/MyDrive/gogolou_2026_notch-main"

name_path = "/add/path/here/gogolou_2026_notch-main"

# ── Derived paths (no changes needed below this line) ─────────────────────────
data_path  = name_path + "/empirical_data"
output_dir = Path(name_path) / "reproduced_results"
output_dir.mkdir(parents=True, exist_ok=True)

sys.path.append(name_path + '/src/')

print(f"Data path:   {data_path}")
print(f"Output dir:  {output_dir}")
print(f"Source path: {name_path + '/src/'}")

**Importing and processing empirical data**

In [ ]:
# Find all CSV files in the directory
csv_files = glob.glob(os.path.join(data_path, "*.csv"))
print(f"Found {len(csv_files)} CSV files: {csv_files}")

# Load all CSVs into a list of DataFrames
dfs = [pd.read_csv(f) for f in csv_files]
print(f"Loaded {len(dfs)} DataFrames.")

**Export function**

In [8]:
def export(obj, filename, output_dir=output_dir, dpi=300):
    path = output_dir / filename
    if isinstance(obj, pd.DataFrame):
        path = path.with_suffix('.csv')
        obj.to_csv(path, index=False)
    elif isinstance(obj, plt.Figure):
        fig = obj
    else:
        fig = plt.gcf()

    if not isinstance(obj, pd.DataFrame):
        for suffix in ['.png', '.pdf']:
            fig.savefig(path.with_suffix(suffix), dpi=dpi, bbox_inches='tight')
        print(f"Saved: {path}.png/.pdf")
    else:
        print(f"Saved: {path}")

**Calculate neural, progenitor, and glial fractions for all replicates**

In [ ]:
from compile_data import calculate_fractions, calculate_bio_rep_means

df_neural, df_progenitors_glial = calculate_fractions(dfs)

export(df_neural, 'hucd_tech_reps_fractions')
export(df_progenitors_glial, 'sox10_s100b_tech_reps_fractions')

bio_rep_means = calculate_bio_rep_means(df_neural, df_progenitors_glial)

# Keep the dict for plotting
bio_dfs_dict = bio_rep_means

# Make the merged version only for export
bio_dfs_merged = (
    bio_rep_means['neural']
    .merge(bio_rep_means['progenitors'], on=['condition', 'day', 'bio_rep'])
    .merge(bio_rep_means['glial'],       on=['condition', 'day', 'bio_rep'])
)
export(bio_dfs_merged, 'bio_rep_fraction_means')

**Extract data for all days**

In [ ]:
from compile_data import extract_data
fractions = extract_data(df_neural, df_progenitors_glial)

**Defining ODEs for compartment model and distance function for ABC**

In [12]:
import construct_model

**Parameter inference using DMSO data**

In [ ]:
import parameter_inference
from datetime import datetime

abc_results_untreated = parameter_inference.parameter_inference(fractions, condition='dmso', num_iterations=1000000)

export(abc_results_untreated, f"abc_results_dmso_{datetime.now():%Y%m%d}")

print(f"\\nResults saved to: {output_dir / f'abc_results_dmso_{datetime.now():%Y%m%d}.csv'}")

**Filter out rejected parameter sets**

In [ ]:
from plotting import filter_accepted_thetas, marginal_distributions

theta_accepted_dmso = filter_accepted_thetas(abc_results_untreated, 0.002)

**Plotting simulated over empirical data, medians only, Untreated**

**Figure 5D**

In [ ]:
from plotting import sim_over_emp_median
sim_emp_median = sim_over_emp_median('dmso', df_neural, df_progenitors_glial, bio_dfs_dict, theta_accepted_dmso, canonical_days=[9,15,22])

export(sim_emp_median, f"Simulated over empirical data, medians only, Untreated")

**Plotting initial k_diff for all models, Untreated**

**Figure 5E**

In [ ]:
from plotting import differentiation_rate_all_models

violin_plot_differentiation_rate = differentiation_rate_all_models('dmso', theta_accepted_dmso)

export(violin_plot_differentiation_rate, f"Differentiation rate for all models, Untreated")

**Plotting k_diff gradient for all models, Untreated**

**Figure 5F**

In [ ]:
from plotting import differentiation_rate_gradient_all_models

violin_plot_differentiation_rate_gradient = differentiation_rate_gradient_all_models('dmso', theta_accepted_dmso)

export(violin_plot_differentiation_rate_gradient, f"Differentiation rate gradient for all models, Untreated")

**Plotting initial neural vs glial bias for all models, Untreated**

**Figure 5G**

In [ ]:
from plotting import initial_neural_vs_glial_bias_all_models

violin_plot_bias = initial_neural_vs_glial_bias_all_models('dmso', theta_accepted_dmso)

export(violin_plot_bias,f"Cross-lineage initial bias comparison across all models, Untreated")

**Plotting initial neural vs glial bias gradient for all models, Untreated**

**Figure 5H**

In [ ]:
from plotting import neural_vs_glial_bias_gradient_all_models

violin_plot_bias_gradient = neural_vs_glial_bias_gradient_all_models('dmso', theta_accepted_dmso)

export(violin_plot_bias_gradient, f"Cross-lineage initial bias comparison across all models, Untreated")

**Cross-lineage JS stats, Untreated**

In [ ]:
from statistical_analysis import _compute_js, compare_posteriors, cross_lineage_js

cross_lineage_test = cross_lineage_js(theta_accepted_dmso, 'dmso')

export(cross_lineage_test, 'Cross-lineage, JS and permutation, Untreated')

**Descriptive statistics, Untreated**

In [ ]:
from statistical_analysis import descriptive_statistics

abc_results_dmso_descriptive_stats = descriptive_statistics(theta_accepted_dmso, 'dmso')

export(abc_results_dmso_descriptive_stats, 'Descriptive statistics - dmso')

**Plotting simulated over empirical data, envelopes, Untreated**

**Figure S3(A-D)**

In [ ]:
from plotting import sim_over_emp_envelope

figs = sim_over_emp_envelope(
    'dmso', df_neural, df_progenitors_glial,
    bio_dfs_dict, theta_accepted_dmso,
    canonical_days=[9, 15, 22], envelope_size=50
)

for (k_diff_type, prob_type), fig in figs.items():
    export(fig, f"Simulated over empirical data, envelopes for model: k_diff: {k_diff_type}, prob: {prob_type}, Untreated")

**Marginal distributions for DMSO**

**Figure S3E**

In [ ]:
from plotting import marginal_distributions

marginals = marginal_distributions(theta_accepted_dmso, 'dmso')

export(marginals, f"Marginal distributions for Untreated")

**Calculate mean of inferred p_initial medians across the 4 models to use for ABC against DAPT**

In [ ]:
from parameter_inference import calculate_p_initial

inferred_p_initial = calculate_p_initial(theta_accepted_dmso)

print(inferred_p_initial)

**p_initial was fixed @ 0.4 for when argument condition='dapt' is passed in parameter_inference.parameter_inference**

**ABC against Notch inhibited data**

In [ ]:
abc_results_notch_inhibited = parameter_inference.parameter_inference(fractions, condition='dapt', num_iterations=5000)

export(abc_results_notch_inhibited, f"abc_results_dapt_{datetime.now():%Y%m%d}")

print(f"\\nResults saved to: {output_dir / f'abc_results_dapt_{datetime.now():%Y%m%d}.csv'}")

**Filter out rejected parameter sets**

In [ ]:
theta_accepted_dapt = filter_accepted_thetas(abc_results_notch_inhibited, 0.002)

**Plotting simulated over empirical data, medians only, Notch inhibited**

**Figure 5J**

In [ ]:
from plotting import sim_over_emp_median
sim_emp_median = sim_over_emp_median('dapt', df_neural, df_progenitors_glial, bio_dfs_dict, theta_accepted_dapt, canonical_days=[9,15,22])

export(sim_emp_median, f"Simulated over empirical data, medians only, Notch inhibited")

**Plotting paramater marginals, Untreated vs Notch inhibited**

**Figure 5(K-P)**

In [ ]:
#concatenate filtered abc results for untreated and notch inhibited
theta_accepted = pd.concat([theta_accepted_dmso, theta_accepted_dapt])

from plotting import cross_condition_marginal_comparison_for_all_models

figs = cross_condition_marginal_comparison_for_all_models(theta_accepted)

for param, fig in figs.items():
    export(fig, f"Cross-condition comparison of {param}")


**Cross-lineage JS stats, Notch inhibited**

In [ ]:
from statistical_analysis import _compute_js, compare_posteriors, cross_lineage_js

cross_lineage_test = cross_lineage_js(theta_accepted_dapt, 'dapt')

export(cross_lineage_test, 'Cross-lineage, JS and permutation, Notch inhibited')

**Cross-condition JS stats**

In [ ]:
from statistical_analysis import _compute_js, compare_posteriors, cross_condition_js

cross_condition_test = cross_condition_js(theta_accepted)

export(cross_condition_test, 'Cross-condition test for all parameters - JS and permutation')

**Marginal distributions for DAPT**

**Figure S4E**

In [ ]:
marginals = marginal_distributions(theta_accepted_dapt, 'dapt')

export(marginals, f"Marginal distributions for Notch inhibited")

**Plotting simulated over empirical data, envelopes, Notch inhibited**

**Figure S4(A-D)**

In [ ]:
from plotting import sim_over_emp_envelope

figs = sim_over_emp_envelope(
    'dapt', df_neural, df_progenitors_glial,
    bio_dfs_dict, theta_accepted_dapt,
    canonical_days=[9, 15, 22], envelope_size=50
)

for (k_diff_type, prob_type), fig in figs.items():
    export(fig, f"Simulated over empirical data, envelopes for model: k_diff: {k_diff_type}, prob: {prob_type}, Notch inhibited")

**End**